In [1]:
import pandas as pd
import numpy as np

## 1. Load and audit

In [3]:
print("BOLLYWOOD MOVIE DATASET")
print("=" * 40)

df = pd.read_csv('DataSet.csv')

print(f"Total movies: {len(df):,}")
print(f"Total columns: {df.shape[1]}")
print(f"Duplicate rows: {df.duplicated().sum():,}")
print(f"Missing values: {df.isna().sum().sum():,}")

print("\nColumn names:")
print(df.columns.tolist())

BOLLYWOOD MOVIE DATASET
Total movies: 1,698
Total columns: 14
Duplicate rows: 0
Missing values: 0

Column names:
['Movie Name', 'Release Period', 'Whether Remake', 'Whether Franchise', 'Genre', 'New Actor', 'New Director', 'New Music Director', 'Lead Star', 'Director', 'Music Director', 'Number of Screens', 'Revenue(INR)', 'Budget(INR)']


In [4]:
data_quality = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Missing Values": df.isna().sum(),
    "Missing (%)": (df.isna().mean() * 100).round(2),
    "Unique Values": df.nunique()
})

display(data_quality.sort_values("Missing Values", ascending=False))

,Data Type,Missing Values,Missing (%),Unique Values
Movie Name,str,0,0.0,1698
Release Period,str,0,0.0,2
Whether Remake,str,0,0.0,2
Whether Franchise,str,0,0.0,2
Genre,str,0,0.0,14
New Actor,str,0,0.0,2
New Director,str,0,0.0,2
New Music Director,str,0,0.0,2
Lead Star,str,0,0.0,761
Director,str,0,0.0,1045


In [5]:
df = df.drop_duplicates()
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Movie Name          1698 non-null   str  
 1   Release Period      1698 non-null   str  
 2   Whether Remake      1698 non-null   str  
 3   Whether Franchise   1698 non-null   str  
 4   Genre               1698 non-null   str  
 5   New Actor           1698 non-null   str  
 6   New Director        1698 non-null   str  
 7   New Music Director  1698 non-null   str  
 8   Lead Star           1698 non-null   str  
 9   Director            1698 non-null   str  
 10  Music Director      1698 non-null   str  
 11  Number of Screens   1698 non-null   int64
 12  Revenue(INR)        1698 non-null   int64
 13  Budget(INR)         1698 non-null   int64
dtypes: int64(3), str(11)
memory usage: 185.8 KB
None


## 2. Remove duplicates and implausible values

In [7]:
initial_len = len(df)
df = df[(df['Budget(INR)'] > 0) & (df['Revenue(INR)'] >= 0) & (df['Number of Screens'] > 0)]
df = df.drop_duplicates()
print(f"Removed {initial_len - len(df)} implausible/duplicate rows.")

Removed 0 implausible/duplicate rows.


## 3. Encode categorical variables

In [9]:
holiday_release = df["Release Period"].map({'Holiday': 1, 'Normal': 0}).to_frame(name="Holiday_Release").astype(bool)
print(holiday_release.info())

is_remake = df[["Whether Remake"]].replace({'Yes': 1, 'No': 0}).astype(bool)
print(is_remake.info())

is_Franchise = df[["Whether Franchise"]].replace({'Yes': 1, 'No': 0}).astype(bool)
print(is_Franchise.info())

<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 1 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Holiday_Release  1698 non-null   bool 
dtypes: bool(1)
memory usage: 1.8 KB
None
<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 1 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Whether Remake  1698 non-null   bool 
dtypes: bool(1)
memory usage: 1.8 KB
None
<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 1 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Whether Franchise  1698 non-null   bool 
dtypes: bool(1)
memory usage: 1.8 KB
None


In [10]:

newcrew = df[["New Actor", "New Director", "New Music Director"]].replace({'Yes': 1, 'No': 0}).astype(bool)
print(newcrew.info())

<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   New Actor           1698 non-null   bool 
 1   New Director        1698 non-null   bool 
 2   New Music Director  1698 non-null   bool 
dtypes: bool(3)
memory usage: 5.1 KB
None


In [11]:
genres = pd.get_dummies(df[["Genre"]], columns=["Genre"], drop_first=True, dtype=bool)
print(genres.info())

<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Genre_adult         1698 non-null   bool 
 1   Genre_animation     1698 non-null   bool 
 2   Genre_comedy        1698 non-null   bool 
 3   Genre_documentary   1698 non-null   bool 
 4   Genre_drama         1698 non-null   bool 
 5   Genre_fantasy       1698 non-null   bool 
 6   Genre_horror        1698 non-null   bool 
 7   Genre_love_story    1698 non-null   bool 
 8   Genre_masala        1698 non-null   bool 
 9   Genre_mythological  1698 non-null   bool 
 10  Genre_rom_com       1698 non-null   bool 
 11  Genre_suspense      1698 non-null   bool 
 12  Genre_thriller      1698 non-null   bool 
dtypes: bool(13)
memory usage: 21.7 KB
None


## 4. Derived columns (Movie_Scale, Revenue Per Screen, ROI)

In [13]:

# We are using a normalized log scale for budget and screens to create a composite score that is then mapped to a 1-10 scale.

w_budget = 0.5  # Budget Weight : equal to screens weight; the mediation analysis in 4.LinearRegression.ipynb (~50% of budget's effect flowing through screens) supports this choice.
w_screens = 0.5 # Screens Weight

log_budget = np.log1p(df["Budget(INR)"])
log_screens = np.log1p(df["Number of Screens"])


norm_budget = (log_budget - log_budget.min()) / (log_budget.max() - log_budget.min())
norm_screens = (log_screens - log_screens.min()) / (log_screens.max() - log_screens.min())

composite = (w_budget * norm_budget) + (w_screens * norm_screens)

movie_scale = (1 + 9 * composite).round(3).to_frame(name="Movie_Scale")
print(movie_scale.info())


<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 1 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Movie_Scale  1698 non-null   float64
dtypes: float64(1)
memory usage: 13.4 KB
None


In [14]:
rps = (df["Revenue(INR)"] / df["Number of Screens"]).round(1).to_frame(name="Revenue Per Screen")
print(rps.info())

<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 1 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Revenue Per Screen  1698 non-null   float64
dtypes: float64(1)
memory usage: 13.4 KB
None


In [15]:
# our final point of interest : the Return On Interest

roi = (df["Revenue(INR)"] / df["Budget(INR)"]).round(5).to_frame(name="ROI")
print(roi.info())

<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   ROI     1698 non-null   float64
dtypes: float64(1)
memory usage: 13.4 KB
None


## 5. Assemble and write CleanedDataSet.csv

In [17]:
## Appending all the cleaned columns

df_clean = pd.concat([df[["Movie Name","Lead Star", "Director", "Music Director"]], is_remake, is_Franchise, holiday_release, genres, newcrew, df[["Budget(INR)", "Revenue(INR)", "Number of Screens"]], rps, movie_scale, roi], axis=1)

df_clean = df_clean.rename(columns={
    'Whether Remake': 'Remake',
    'Whether Franchise': 'Franchise',
    'Holiday_Release': 'Holiday',
    'New Actor': 'New_Actor',
    'New Director': 'New_Director',
    'New Music Director': 'New_Music_Director',
    'Budget(INR)': 'Budget',
    'Revenue(INR)': 'Revenue'
})
print(df_clean.info())

<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 29 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Movie Name          1698 non-null   str    
 1   Lead Star           1698 non-null   str    
 2   Director            1698 non-null   str    
 3   Music Director      1698 non-null   str    
 4   Remake              1698 non-null   bool   
 5   Franchise           1698 non-null   bool   
 6   Holiday             1698 non-null   bool   
 7   Genre_adult         1698 non-null   bool   
 8   Genre_animation     1698 non-null   bool   
 9   Genre_comedy        1698 non-null   bool   
 10  Genre_documentary   1698 non-null   bool   
 11  Genre_drama         1698 non-null   bool   
 12  Genre_fantasy       1698 non-null   bool   
 13  Genre_horror        1698 non-null   bool   
 14  Genre_love_story    1698 non-null   bool   
 15  Genre_masala        1698 non-null   bool   
 16  Genre_mythologica

In [18]:
print(df_clean.head(2))
## Writing back to a new CSV file for further analysis and model building.
df_clean.to_csv('CleanedDataSet.csv', index=False)

   Movie Name   Lead Star      Director Music Director  Remake  Franchise  \
0  Qaidi Band  Aadar Jain  Habib Faisal   Amit Trivedi   False      False   
1   Chaatwali  Aadil Khan    Aadil Khan   Babloo Ustad   False      False   

   Holiday  Genre_adult  Genre_animation  Genre_comedy  ...  Genre_thriller  \
0     True        False            False         False  ...           False   
1     True         True            False         False  ...           False   

   New_Actor  New_Director  New_Music_Director     Budget  Revenue  \
0       True         False               False  210000000  4500000   
1       True          True                True    1000000  1075000   

   Number of Screens  Revenue Per Screen  Movie_Scale      ROI  
0                800              5625.0        7.803  0.02143  
1                  1           1075000.0        1.576  1.07500  

[2 rows x 29 columns]


## 6. Output verification (against the raw data)

The cleaned CSV is read back and checked row-by-row against `DataSet.csv`: same row count, identical
budget/revenue/screens, and the `Holiday` flag consistent with the raw `Release Period` label.

In [20]:
chk_src = pd.read_csv('DataSet.csv')
chk_out = pd.read_csv('CleanedDataSet.csv')

chk = chk_src[["Movie Name", "Budget(INR)", "Revenue(INR)", "Number of Screens",
               "Release Period"]].merge(
      chk_out[["Movie Name", "Budget", "Revenue", "Number of Screens", "Holiday"]],
      on="Movie Name", suffixes=("_raw", "_clean"))
assert len(chk) == len(chk_src) == len(chk_out), "row count mismatch"
assert (chk["Budget(INR)"] == chk["Budget"]).all(), "budget mismatch"
assert (chk["Revenue(INR)"] == chk["Revenue"]).all(), "revenue mismatch"
assert (chk["Number of Screens_raw"] == chk["Number of Screens_clean"]).all(), "screens mismatch"
assert ((chk["Release Period"] == "Holiday") == chk["Holiday"]).all(), "holiday flag mismatch"
print(f"output verification passed: all {len(chk)} rows match the raw data")

output verification passed: all 1698 rows match the raw data
